# IOAI — 2026 Contest Leopard Cat Reid (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-leopard-cat-reid/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Spot the Spots — 표범살쾡이 개체 식별 (APOAI 2026)

카메라트랩 이미지에서 **54마리 개별 표범살쾡이**(*Prionailurus bengalensis*)를 세밀분류(Re-ID)한다.
점박이 무늬가 개체마다 고유하다. 주야간(적외선) 조명·배경·자세 변화에 강인해야 한다. `data/train/`(이미지)+
`data/train.csv`(filename,label 0~53)로 학습하고 `data/test/`(라벨 없음)를 예측해 `submission.csv`(filename,label).
채점: **accuracy**. 사전학습 backbone 허용(ConvNeXt/ResNet101/Swin — torchvision 기본 가중치).

In [ ]:
import pandas as pd, os
train = pd.read_csv("data/train.csv")            # filename, label (0..53)
test_list = pd.read_csv("data/test_list.csv")["filename"].tolist()
print("train", len(train), "| classes", train.label.nunique(), "| test", len(test_list))

In [ ]:
# 베이스라인: 최빈 개체로 예측 — accuracy 낮음
maj = int(train.label.mode()[0])
pd.DataFrame({"filename": test_list, "label": [maj]*len(test_list)}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(test_list))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)